# Final models and frozen results

This notebook reproduces the **September17,2026 research snapshot** offline. It does not download new data. The main model is the final Gaussian; helpers are corrected polling, the actual research Student-t model and mean-only blends.

**What is predicted?** Each state/contest's final D−R vote margin, in percentage points. Positive means favor Democrats. P(D) is a win probability. Total seats include continuing seats; point totals count positive means, expected totals sum probabilities. Read [the model specification](../docs/MODEL.md) for the equations.

The Student helper retains its earlier research architecture; it differs in priors/correlation as well as distribution. Its saved converged output is shown here; notebook03 reruns the sampler.

In [2]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p/'election_lab.py').exists())
sys.path.insert(0, str(ROOT))
import election_lab as lab
from model_labels import display, model_label
pd.set_option('display.max_rows', 40)


## 1. Reproduce numerical forecasts

Recompute all15 Gaussian posteriors and the non-Bayesian historical bias, then translate the Gaussian means. Original posteriors and joint seat simulations are checked. Historical component fits use earlier cycles only. Full Gaussian Bayesian covariance is retained for every blend.

In [3]:
RUN = lab.run_logged(lab.reproduce)
tables = lab.display_tables(RUN)
print('Saved:', RUN.relative_to(ROOT))

Run log: cache/logs/reproduce_20260921T185658.425890Z.txt
Saved: cache/runs/reproduction/20260921T185658.521444Z


## 2. Historical comparison

Show five recent cycles (2016–2024),140 contests per horizon. MAE is margin error in percentage points; lower is better. Brier scores probabilities; lower is better. Correct calls pool contests; other metrics average cycles equally. Non-Bayesian probabilities start only after three earlier evaluated cycles, so `probability_cycles` can be smaller—do not compare unequal probability coverage silently.

In [4]:
MODELS = ['Bayesian', 'Non-Bayesian corrected', 'Student-t research helper', 'Corrected 20%', 'Corrected 50%']
summary = tables['summary']
display(summary[summary.first_cycle.eq(2016) & summary.model.isin(MODELS)][['scenario','model','cycles','n','correct','absolute_error_pp','probability_cycles','brier','coverage70']].round(4))
print('Probability comparison on the common 2018–2024 range:')
display(summary[summary.first_cycle.eq(2018) & summary.model.isin(MODELS)][['scenario','model','n','brier','coverage70']].round(4))

,scenario,model,cycles,n,correct,absolute_error_pp,probability_cycles,brier,coverage70
26,matched_live,Gaussian Bayesian,5,140,128,6.3067,5,0.0555,0.7821
28,matched_live,Corrected 20%,5,140,129,6.2772,5,0.0560,0.8097
32,matched_live,Corrected 50%,5,140,129,6.5649,5,0.0585,0.7964
34,matched_live,Non-Bayesian corrected,5,140,127,7.4722,4,0.0785,0.8970
38,matched_live,Student-t research helper,5,140,129,6.8011,5,0.0579,0.6599
39,oct31,Gaussian Bayesian,5,140,132,5.2060,5,0.0504,0.7317
41,oct31,Corrected 20%,5,140,133,5.0349,5,0.0483,0.7548
45,oct31,Corrected 50%,5,140,133,4.9394,5,0.0460,0.7486
47,oct31,Non-Bayesian corrected,5,140,133,5.2202,4,0.0372,0.8532
51,oct31,Student-t research helper,5,140,131,5.3574,5,0.0490,0.6152


Probability comparison on the common 2018–2024 range:


,scenario,model,n,brier,coverage70
52,matched_live,Gaussian Bayesian,111,0.0434,0.7708
54,matched_live,Corrected 20%,111,0.0435,0.8052
58,matched_live,Corrected 50%,111,0.0454,0.7972
60,matched_live,Non-Bayesian corrected,111,0.0785,0.8970
64,matched_live,Student-t research helper,111,0.0436,0.6352
65,oct31,Gaussian Bayesian,111,0.0389,0.7767
67,oct31,Corrected 20%,111,0.0367,0.8056
71,oct31,Corrected 50%,111,0.0343,0.7892
73,oct31,Non-Bayesian corrected,111,0.0372,0.8532
77,oct31,Student-t research helper,111,0.0361,0.6311


## 3. Current total seats

All values below are frozen September17 inputs. A51D point count does not imply more than50% control probability: close races and correlations determine the joint seat distribution. The non-Bayesian helper's own control odds assume independent states; blends retain Gaussian Bayesian dependence.

In [5]:
seats = tables['seats'].query('model in @MODELS').copy()
seats['D control %'] = 100*seats.p_D_control
seats['70% D seats'] = seats.D_lo70.astype(int).astype(str)+'–'+seats.D_hi70.astype(int).astype(str)
display(seats[['model','point_D','point_R','expected_D','D control %','70% D seats','method']].round(3))

,model,point_D,point_R,expected_D,D control %,70% D seats,method
81,Gaussian Bayesian,51,49,50.426,48.270,49–52,Bayesian joint covariance
84,Corrected 20%,51,49,50.166,41.833,49–52,Bayesian joint covariance
87,Corrected 50%,51,49,49.717,32.617,48–51,Bayesian joint covariance
92,Non-Bayesian corrected,49,51,48.463,17.300,46–51,Independent-state approximation; helper only
181,Student-t research helper,51,49,51.256,66.247,49–53,Research MCMC; separate earlier architecture


## 4. State margins and probabilities

All35 current contests remain in the table, including no-poll cases. Models produce margins directly; probabilities are derived using their uncertainty. Independent-candidate/caucus and election-rule assumptions remain as documented.

In [6]:
print('D−R margins (percentage points)')
display(tables['margins'][MODELS].round(2))
print('Democratic win probability (%)')
display(tables['probabilities'][MODELS].round(1))

D−R margins (percentage points)


model,Gaussian Bayesian,Non-Bayesian corrected,Student-t research helper,Corrected 20%,Corrected 50%
State,,,,,
AK,-1.71,1.04,1.49,-1.16,-0.33
AL,-21.90,-21.69,-18.50,-21.86,-21.80
AR,-20.75,-23.32,-11.66,-21.27,-22.04
CO,17.48,5.95,14.45,15.17,11.71
DE,24.99,15.52,25.92,23.09,20.25
FL (special),-6.12,-10.48,-5.29,-6.99,-8.30
GA,5.76,4.62,6.76,5.53,5.19
IA,-3.14,-5.20,-2.60,-3.55,-4.17
ID,-25.95,-29.76,-26.73,-26.71,-27.86


Democratic win probability (%)


model,Gaussian Bayesian,Non-Bayesian corrected,Student-t research helper,Corrected 20%,Corrected 50%
State,,,,,
AK,38.8,53.3,59.4,42.4,47.8
AL,0.0,4.2,1.8,0.0,0.0
AR,0.2,3.1,9.5,0.1,0.1
CO,92.8,68.2,88.9,89.8,83.6
DE,99.9,89.2,98.9,99.8,99.5
FL (special),17.8,20.2,21.9,14.6,10.5
GA,83.0,64.4,86.1,82.0,80.4
IA,29.9,33.9,31.0,27.6,24.2
ID,0.0,0.9,0.5,0.0,0.0


## 5. Each historical cycle versus actuals

Historical chamber totals include fixed completion of unmodeled contests; those completions are not counted as modeled state predictions.

In [7]:
all_seats = pd.read_parquet(RUN/'seats.parquet')
for horizon in ['matched_live','oct31']:
    g = all_seats[(all_seats.scenario==horizon)&(all_seats.cycle<2026)&all_seats.model.isin(MODELS)]
    print(horizon, 'expected Democratic seats')
    actual = g[g.model.eq('Bayesian')].set_index('cycle')[['actual_D','unmodeled_contested']]
    display(actual.join(g.pivot(index='cycle',columns='model',values='expected_D')).round(2))

matched_live expected Democratic seats


,actual_D,unmodeled_contested,Gaussian Bayesian,Corrected 20%,Corrected 50%,Non-Bayesian corrected,Student-t research helper
cycle,,,,,,,
2012,55.0,2,50.77,50.82,50.87,NaN,51.06
2014,46.0,9,51.53,51.82,52.25,NaN,51.77
2016,48.0,5,49.99,50.06,50.18,NaN,49.88
2018,47.0,5,47.65,47.78,48.00,46.75,48.25
2020,50.0,8,49.49,49.28,48.90,48.37,49.77
2022,51.0,9,52.46,52.47,52.50,52.28,52.19
2024,47.0,6,48.05,47.88,47.55,46.02,48.52


oct31 expected Democratic seats


,actual_D,unmodeled_contested,Gaussian Bayesian,Corrected 20%,Corrected 50%,Non-Bayesian corrected,Student-t research helper
cycle,,,,,,,
2012,55.0,2,51.44,51.33,51.16,NaN,51.23
2014,46.0,9,49.00,49.10,49.28,NaN,49.04
2016,48.0,5,50.20,50.16,50.11,NaN,50.26
2018,47.0,5,46.99,47.01,47.03,46.62,47.14
2020,50.0,8,48.96,48.80,48.57,48.61,49.05
2022,51.0,9,51.51,51.48,51.46,51.32,51.39
2024,47.0,6,47.97,47.73,47.39,46.59,48.20


## Interpretation

The Gaussian is the designated main reference. Corrected-polling blends improved late-horizon historical results but not all cycles/subgroups; larger September blends hurt. Student is a tail-sensitivity helper. Repeated architecture exploration and few independent cycles make all comparisons exploratory. The next notebook isolates blend weights without changing covariance.

## Saved reports

[Read the published report](../outputs/reports/experiments/reproduction.md) · [All outputs and dated reports](../outputs/README.md)
